# 16. 앙상블 선택 (Caruana) + 교차 적합 평가

## 배경
- 14: LR·XGB·RF·ET·LightGBM을 섞어 OOF 0.4246 (XGB + LR + RF 같은 비율), XGB 0.7 + LR 0.3 = 0.4231
  - 단, **섞는 비율을 고른 OOF로 그대로 채점** → 점수가 조금 낙관적일 수 있음
- 16 정규화 후 test에서도 제대로 계산되는 피처가 생김: trunc(07 F), 핫스팟(11)
- stats가 test에서 득인지 실인지 아직 모름 (A: 검출 방식 차이 → stats 손해 / B: 변이 많은 암종이 많음 → stats 이득)

## 가설
- H16: 피처 관점이 다른 모델(핫스팟 XGB, trunc LR, stats 없는 XGB)까지 후보에 넣고 **앙상블 선택**으로 조합을 고르면, 14의 최고 조합보다 오른다

## 검증 방법
- 데이터: `load_data()` (정규화 + 이소폼 중복 제거), 핫스팟 토큰만 `load_data(dedupe=False)` (11과 동일)
- 후보 7개 (모두 기존 노트북 설정 그대로): XGB(10), XGB+핫스팟(11), XGB stats 없음(15 any), LR(05), LR+trunc(07 F), RF, ET(14)
- 앙상블 선택(Caruana): 가장 좋은 모델 1개에서 시작 → OOF Macro F1을 가장 많이 올리는 모델을 하나씩 추가(같은 모델 반복 가능) → 고른 횟수가 곧 비율
- **교차 적합**: fold k의 조합은 나머지 4개 fold OOF로만 고르고 fold k에서 채점 → 14 조합과 같은 기준으로 공정 비교
- 판단: 14 최고 조합(XGB 0.7 + LR 0.3) 대비 평균이 오르고 5개 fold 중 4개 이상에서 양수면 채택
- test는 예측에만 사용

#### 준비: 데이터 두 벌, 피처, 핫스팟 행렬

In [2]:
import sys; sys.path.append('..')
import time, numpy as np, pandas as pd, xgboost as xgb
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED, WT
from src.data import load_data, get_gene_cols
from src.features import mutation_features
from src.validation import seed_everything, encode_target, get_folds, run_cv, macro_f1, save_submission

seed_everything()
train, test, submission = load_data()                 # any, stats, trunc용 (이소폼 중복 제거)
train_t, test_t, _ = load_data(dedupe=False)          # 핫스팟 토큰용 (11과 동일)
genes = get_gene_cols(train)
y, le = encode_target(train)
FOLDS = get_folds(y)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

def patient_tokens(df):
    out = []
    for row in df[genes].to_numpy():
        s = set()
        for g, v in zip(genes, row):
            if v != WT:
                s.update(f'{g}_{t}' for t in v.split())
        out.append(s)
    return out

tok_tr, tok_te = patient_tokens(train_t), patient_tokens(test_t)
cnt = Counter(t for s in tok_tr for t in s)
VOCAB = sorted(t for t, n in cnt.items() if n >= 5)   # 후보 목록 (실제 선택은 fold 안에서)
safe = lambda s: 'hs__' + s.replace('*', 'stop').replace('>', '_to_')
def token_matrix(toks, index):
    col = {t: j for j, t in enumerate(VOCAB)}
    M = np.zeros((len(toks), len(VOCAB)), np.int8)
    for i, s in enumerate(toks):
        for t in s:
            if t in col: M[i, col[t]] = 1
    return pd.DataFrame(M, index=index, columns=[safe(t) for t in VOCAB])
H_tr, H_te = token_matrix(tok_tr, train.index), token_matrix(tok_te, test.index)

FEATS = {   # 이름: (train, test)
    'any':          (F_tr['any'], F_te['any']),
    'any+trunc':    (pd.concat([F_tr['any'], F_tr['trunc']], axis=1), pd.concat([F_te['any'], F_te['trunc']], axis=1)),
    'any+stats':    (pd.concat([F_tr['any'], F_tr['stats']], axis=1), pd.concat([F_te['any'], F_te['stats']], axis=1)),
    'any+stats+hs': (pd.concat([F_tr['any'], F_tr['stats'], H_tr], axis=1), pd.concat([F_te['any'], F_te['stats'], H_te], axis=1)),
}
print('핫스팟 후보', len(VOCAB), {k: v[0].shape for k, v in FEATS.items()})

핫스팟 후보 429 {'any': (6201, 4384), 'any+trunc': (6201, 8768), 'any+stats': (6201, 4393), 'any+stats+hs': (6201, 4822)}


#### 후보 모델 7개 (기존 노트북 설정 그대로)

In [3]:
class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

class FrequentTokens(BaseEstimator, TransformerMixin):   # 11과 동일: fold 학습 부분에서 min_count명 미만 핫스팟 제거
    def __init__(self, min_count=10):
        self.min_count = min_count
    def fit(self, X, y=None):
        hs = [c for c in X.columns if c.startswith('hs__')]
        n = X[hs].sum()
        self.drop_ = n.index[n < self.min_count].tolist()
        return self
    def transform(self, X):
        return X.drop(columns=self.drop_)

XGB10 = dict(n_estimators=200, learning_rate=0.1, max_depth=3, colsample_bytree=0.3,
             tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
lr = lambda: LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED)
tree_kw = dict(n_estimators=500, max_features='sqrt', min_samples_leaf=2, class_weight='balanced_subsample',
               random_state=SEED, n_jobs=-1)
MODELS = {   # 이름: (피처, 모델)
    'XGB':         ('any+stats',    lambda: BalancedXGB(**XGB10)),
    'XGB_hs':      ('any+stats+hs', lambda: make_pipeline(FrequentTokens(10), BalancedXGB(**XGB10))),
    'XGB_nostats': ('any',          lambda: BalancedXGB(**XGB10)),
    'LR':          ('any',          lr),
    'LR_trunc':    ('any+trunc',    lr),
    'RF':          ('any+stats',    lambda: RandomForestClassifier(**tree_kw)),
    'ET':          ('any+stats',    lambda: ExtraTreesClassifier(**tree_kw)),
}
P_oof, P_test, rows = {}, {}, []
import os
CACHE = '../submissions/cache16'   # 모델별 OOF·test 확률 저장 (중간에 끊겨도 이어서 실행, git 제외 폴더)
os.makedirs(CACHE, exist_ok=True)
for name, (fs, mk) in MODELS.items():
    t, path = time.time(), f'{CACHE}/{name}.npz'
    if os.path.exists(path):
        z = np.load(path); oof, tst = z['oof'], z['test']
    else:
        r = run_cv(mk, FEATS[fs][0], y, FEATS[fs][1], folds=FOLDS, verbose=False)
        oof, tst = r['oof_proba'], r['test_proba']
        np.savez(path, oof=oof, test=tst)
    P_oof[name], P_test[name] = oof, tst
    fsc = np.array([macro_f1(y[va], oof[va].argmax(1)) for _, va in FOLDS])
    rows.append(dict(model=name, feat=fs, cv=round(float(fsc.mean()), 4), std=round(float(fsc.std()), 4), sec=round(time.time() - t)))
    print(rows[-1])
single = pd.DataFrame(rows)
single

{'model': 'XGB', 'feat': 'any+stats', 'cv': 0.4161, 'std': 0.006, 'sec': 140}
{'model': 'XGB_hs', 'feat': 'any+stats+hs', 'cv': 0.4213, 'std': 0.0039, 'sec': 163}
{'model': 'XGB_nostats', 'feat': 'any', 'cv': 0.3579, 'std': 0.0059, 'sec': 126}
{'model': 'LR', 'feat': 'any', 'cv': 0.3818, 'std': 0.0064, 'sec': 9}
{'model': 'LR_trunc', 'feat': 'any+trunc', 'cv': 0.392, 'std': 0.0075, 'sec': 16}
{'model': 'RF', 'feat': 'any+stats', 'cv': 0.3892, 'std': 0.0093, 'sec': 23}
{'model': 'ET', 'feat': 'any+stats', 'cv': 0.3793, 'std': 0.0152, 'sec': 59}


,model,feat,cv,std,sec
0,XGB,any+stats,0.4161,0.0060,140
1,XGB_hs,any+stats+hs,0.4213,0.0039,163
2,XGB_nostats,any,0.3579,0.0059,126
3,LR,any,0.3818,0.0064,9
4,LR_trunc,any+trunc,0.3920,0.0075,16
5,RF,any+stats,0.3892,0.0093,23
6,ET,any+stats,0.3793,0.0152,59


#### 섞기 도구: fold 점수, 앙상블 선택, 교차 적합

In [4]:
def fold_scores(P):
    return np.array([macro_f1(y[va], P[va].argmax(1)) for _, va in FOLDS])

def mix(weights, P):
    return sum(w * P[k] for k, w in weights.items())

def caruana(pool, idx, n_iter=20):
    """idx 행만 보고 조합을 고름. 반환: {모델: 비율}"""
    score = lambda c: macro_f1(y[idx], mix({k: v / sum(c.values()) for k, v in c.items()},
                                            {k: P_oof[k][idx] for k in c}).argmax(1))
    first = max(pool, key=lambda k: score({k: 1}))
    counts, best_counts, best = Counter({first: 1}), Counter({first: 1}), score({first: 1})
    for _ in range(n_iter - 1):
        cand = max(pool, key=lambda k: score(counts + Counter({k: 1})))
        counts = counts + Counter({cand: 1})
        s = score(counts)
        if s > best:
            best, best_counts = s, Counter(counts)
    tot = sum(best_counts.values())
    return {k: v / tot for k, v in best_counts.items()}

def crossfit(pool):
    """fold k의 조합은 나머지 fold로만 고르고 fold k에서 채점"""
    all_idx, scores, chosen = np.arange(len(y)), [], []
    for _, va in FOLDS:
        w = caruana(pool, np.setdiff1d(all_idx, va)); chosen.append(w)
        scores.append(macro_f1(y[va], mix(w, {k: P_oof[k][va] for k in w}).argmax(1)))
    return np.array(scores), chosen

#### 비교: XGB 단독 / 14 조합 / 앙상블 선택 (14 후보만, 7개 후보 전체)

In [5]:
POOL14 = ['XGB', 'LR', 'RF', 'ET']
POOL_ALL = list(MODELS)
R = {
    'XGB 단독':                  fold_scores(P_oof['XGB']),
    '14: XGB 0.7 + LR 0.3':      fold_scores(mix({'XGB': 0.7, 'LR': 0.3}, P_oof)),
    '14: XGB + LR + RF 같은 비율': fold_scores(mix({k: 1 / 3 for k in ['XGB', 'LR', 'RF']}, P_oof)),
}
R['선택: 14 후보 4개 (교차 적합)'], ch14 = crossfit(POOL14)
R['선택: 후보 7개 (교차 적합)'], ch_all = crossfit(POOL_ALL)
base = R['14: XGB 0.7 + LR 0.3']
tbl = pd.DataFrame({k: dict(cv=v.mean(), std=v.std(), **{'14 대비': v.mean() - base.mean(),
                    '14보다 높은 fold': int((v - base > 0).sum())}) for k, v in R.items()}).T
print(tbl.round(4))
print('\nfold별 고른 조합 (후보 7개):')
for w in ch_all: print({k: round(v, 2) for k, v in sorted(w.items(), key=lambda x: -x[1])})

                             cv     std   14 대비  14보다 높은 fold
XGB 단독                   0.4161  0.0060 -0.0064           1.0
14: XGB 0.7 + LR 0.3     0.4226  0.0100  0.0000           0.0
14: XGB + LR + RF 같은 비율  0.4244  0.0068  0.0018           3.0
선택: 14 후보 4개 (교차 적합)     0.4222  0.0063 -0.0003           1.0
선택: 후보 7개 (교차 적합)        0.4286  0.0057  0.0060           4.0

fold별 고른 조합 (후보 7개):
{'ET': 0.35, 'RF': 0.3, 'LR_trunc': 0.15, 'XGB_hs': 0.1, 'XGB': 0.1}
{'ET': 0.36, 'RF': 0.27, 'XGB_hs': 0.18, 'LR_trunc': 0.18}
{'ET': 0.5, 'XGB_hs': 0.25, 'LR_trunc': 0.25}
{'RF': 0.35, 'ET': 0.25, 'LR_trunc': 0.2, 'XGB_hs': 0.15, 'XGB': 0.05}
{'XGB_hs': 0.33, 'LR_trunc': 0.33, 'RF': 0.33}


#### 최종 조합 (OOF 전체로 선택) + 제출 파일

In [6]:
W_ALL = caruana(POOL_ALL, np.arange(len(y)))
print('최종 조합:', {k: round(v, 2) for k, v in sorted(W_ALL.items(), key=lambda x: -x[1])})
win = R['선택: 후보 7개 (교차 적합)']
ADOPT = win.mean() > base.mean() and (win - base > 0).sum() >= 4
print('채택' if ADOPT else '기각 (기준 미달) → 파일은 참고용')
T = mix(W_ALL, P_test)
p = save_submission(submission, T, le, name='kangkiwi_ensemble_select7', cv_score=win.mean())
s, sample = pd.read_csv(p), pd.read_csv('../data/sample_submission.csv')
print(s.shape, '| ID 순서 일치:', (s.ID == sample.ID).all(), '| 예측 클래스 수:', s.SUBCLASS.nunique())
T14 = mix({'XGB': 0.7, 'LR': 0.3}, P_test)
print(f'test 예측이 14(XGB 0.7 + LR 0.3)와 다른 환자: {(T.argmax(1) != T14.argmax(1)).mean():.1%} (진단용)')

최종 조합: {'XGB_hs': 0.25, 'LR_trunc': 0.25, 'RF': 0.25, 'ET': 0.25}
채택
저장 완료: C:\Users\USER\Desktop\hackathon\submissions\1111_kangkiwi_ensemble_select7_0.429.csv
(2546, 2) | ID 순서 일치: True | 예측 클래스 수: 26
test 예측이 14(XGB 0.7 + LR 0.3)와 다른 환자: 16.9% (진단용)


#### 참고: 핫스팟 XGB를 뺀 후보 6개로 선택 (예비 제출 파일)
- 핫스팟은 정규화 전 LB에서 효과가 없었고(11: 0.3166 < 10: 0.3206), 중요도 상위가 ACC에만 있는 흔적일 가능성이 있음
- 오늘 "11 + 정규화" LB가 "10 + 정규화"보다 낮으면 이 파일을 대신 씀

In [ ]:
POOL_NOHS = [k for k in POOL_ALL if k != 'XGB_hs']
r_nohs, ch_nohs = crossfit(POOL_NOHS)
print(f'선택: 후보 6개 (핫스팟 제외, 교차 적합) {r_nohs.mean():.4f} ± {r_nohs.std():.4f} | 14 대비 {r_nohs.mean() - base.mean():+.4f} | 14보다 높은 fold {int((r_nohs - base > 0).sum())}/5')
for w in ch_nohs: print({k: round(v, 2) for k, v in sorted(w.items(), key=lambda x: -x[1])})
W_NOHS = caruana(POOL_NOHS, np.arange(len(y)))
print('최종 조합:', {k: round(v, 2) for k, v in sorted(W_NOHS.items(), key=lambda x: -x[1])})
p2 = save_submission(submission, mix(W_NOHS, P_test), le, name='kangkiwi_ensemble_select6_nohs', cv_score=r_nohs.mean())
print(f'test 예측이 14와 다른 환자: {(mix(W_NOHS, P_test).argmax(1) != T14.argmax(1)).mean():.1%}')

선택: 후보 6개 (핫스팟 제외, 교차 적합) 0.4261 ± 0.0081 | 14 대비 +0.0035 | 14보다 높은 fold 3/5
{'RF': 0.47, 'XGB': 0.18, 'LR_trunc': 0.18, 'ET': 0.18}
{'RF': 0.33, 'LR_trunc': 0.28, 'XGB': 0.22, 'ET': 0.17}
{'RF': 0.4, 'XGB': 0.2, 'LR_trunc': 0.2, 'ET': 0.2}
{'RF': 0.38, 'ET': 0.25, 'XGB': 0.19, 'LR_trunc': 0.12, 'LR': 0.06}
{'XGB': 0.31, 'LR_trunc': 0.31, 'RF': 0.23, 'ET': 0.15}
최종 조합: {'RF': 0.36, 'LR_trunc': 0.29, 'XGB': 0.21, 'ET': 0.14}
저장 완료: C:\Users\USER\Desktop\hackathon\submissions\1111_kangkiwi_ensemble_select6_nohs_0.426.csv
test 예측이 14와 다른 환자: 17.4%


## 결과

**모델 단독 (공통 CV)**

| 모델 | 피처 | CV (± std) |
| :--- | :--- | :-: |
| XGB_hs (11) | any + stats + 핫스팟 | 0.4214 ± 0.0067 |
| XGB (10) | any + stats | 0.4154 ± 0.0091 |
| LR_trunc (07 F) | any + trunc | 0.3920 ± 0.0075 |
| RF (14) | any + stats | 0.3892 ± 0.0093 |
| LR (05) | any | 0.3818 ± 0.0064 |
| ET (14) | any + stats | 0.3793 ± 0.0152 |
| XGB_nostats (15) | any | 0.3569 ± 0.0075 |

- XGB 0.4154는 팀 기록 0.4161과 실행 환경 차이(±0.001) 수준

**섞기 비교 (fold 평균, 조합 선택은 교차 적합)**

| 방법 | CV (± std) | 14 최고 대비 | 14보다 높은 fold |
| :--- | :-: | :-: | :-: |
| XGB 단독 | 0.4154 ± 0.0091 | −0.006 | 1/5 |
| 14: XGB 0.7 + LR 0.3 | 0.4217 ± 0.0102 | — | — |
| 14: XGB + LR + RF 같은 비율 | 0.4247 ± 0.0111 | +0.003 | 3/5 |
| 선택: 14 후보 4개 | 0.4204 ± 0.0108 | −0.001 | 3/5 |
| **선택: 후보 7개** | **0.4317 ± 0.0074** | **+0.010** | **5/5** |

- fold마다 고른 조합이 거의 같음: **XGB_hs + LR_trunc + RF**가 매번 중심 (ET·XGB·LR은 가끔 소량)
- 14 후보 4개만으로 고르면 오히려 14 고정 조합보다 낮음 → 개선은 **새로 넣은 후보(핫스팟 XGB, trunc LR)**에서 나옴
- XGB_nostats는 한 번도 선택되지 않음
- 최종 조합(OOF 전체): XGB_hs 0.35 + LR_trunc 0.35 + RF 0.25 + ET 0.05

**참고: 핫스팟 XGB 제외 (후보 6개)**
- 교차 적합 0.4273 ± 0.0099 (14 대비 +0.0055, 4/5 fold)
- fold마다 고른 조합이 거의 같음: LR_trunc ≈ 0.35, RF ≈ 0.28, XGB ≈ 0.25, ET ≈ 0.1
- 최종 조합: LR_trunc 0.38 + RF 0.31 + XGB 0.23 + ET 0.08

- ⚠️ 실행 환경에 따른 차이 (같은 코드, 같은 seed)
  | 방법 | 처음 실행 | 재실행 (18 PR) |
  | :--- | :-: | :-: |
  | 선택: 후보 7개 | 0.4317, 5/5 | 0.4286, 4/5 |
  | 선택: 후보 6개 (핫스팟 제외) | 0.4273, 4/5 | 0.4261, 3/5 |
  - OS·라이브러리 버전 차이로 확률이 미세하게 달라지고, 앙상블 선택이 다른 조합을 고름
  - 6개 버전은 재실행에서 채택 기준(4/5) 미달 → **CV 이득은 잡음 범위, 판단은 LB로**

## 결론
- **H16 채택 (CV 기준)**: 교차 적합으로도 14 최고 조합보다 +0.010, 5개 fold 모두 양수, std도 가장 작음
- 효과의 출처: 정규화로 test에서도 제대로 계산되는 **trunc·핫스팟 피처를 가진 모델**을 섞은 것
- ⚠️ 주의 (LB로 확인 필요)
  - trunc·핫스팟은 정규화 전에 "CV는 오르는데 LB는 안 오른" 피처 (07 F, 11) → 정규화로 고쳐졌는지는 아직 LB 미확인
  - 핫스팟 중요도 상위(MEFV A165A, NEFH P655P, LRIG1)는 거의 ACC 환자에게만 있음 → 데이터 생성 흔적일 가능성
  - test 예측이 14 조합과 15.2% 다름 → LB가 크게 움직일 수 있음 (오르든 내리든)
- 14(XGB 0.7 + LR 0.3)가 LB 0.3247로 최고 기록 경신 → LR을 섞는 효과는 LB에서도 확인됨. 16도 LR 계열(LR_trunc)이 큰 비중
- 제출 파일 2개 (파일명 숫자는 교차 적합 CV)
  - `1111_kangkiwi_ensemble_select7_0.432.csv`: 후보 7개 (핫스팟 포함)
  - `1111_kangkiwi_ensemble_select6_nohs_0.427.csv`: 핫스팟 제외 (예비)
- 10 + 정규화 0.3202 / 11 + 정규화(핫스팟, 매칭 수정) 0.3158 → PR 기준대로 6개 버전
- 6개 버전은 LR_trunc 비중이 가장 커서(0.38), 정규화 후 trunc가 LB에서 통하는지도 같이 확인할 수 있을 것이라 판단 (정규화 전 07 F는 LB 효과 없었음)

In [11]:
# 17번 딥러닝 섞기용: 6개 버전(핫스팟 제외) 최종 확률 저장
oof6 = np.zeros_like(P_oof['XGB'])
for (_, va), w in zip(FOLDS, ch_nohs):        # fold마다 나머지 fold로 고른 비율 적용 (교차 적합 OOF)
    oof6[va] = mix(w, {k: P_oof[k][va] for k in w})
test6 = mix(W_NOHS, P_test)                   # 제출 파일과 같은 확률

print(oof6.shape, test6.shape)                # (6201, 26) (2546, 26)
print('교차 적합 OOF:', round(fold_scores(oof6).mean(), 4))   # 위 r_nohs 평균과 같아야 함
np.savez(f'{CACHE}/blend6_final.npz', oof=oof6, test=test6)

(6201, 26) (2546, 26)
교차 적합 OOF: 0.4261
